# Tax Admin — Datadog LLM Observability Experiments

This notebook pulls your dataset from Datadog, runs tax extraction tasks against it,
and evaluates the results using both rule-based and LLM-as-a-judge evaluators.

**Prerequisites:**
- `DD_API_KEY`, `DD_APP_KEY`, `DD_SITE=us5.datadoghq.com` in your environment
- `ANTHROPIC_API_KEY` in your environment
- `pip install ddtrace anthropic`

In [12]:
import os

for var in ["DD_API_KEY", "DD_APP_KEY", "DD_SITE", "ANTHROPIC_API_KEY"]:
    val = os.environ.get(var, "")
    status = f"{val[:12]}..." if val else "NOT SET ⚠️"
    print(f"{var}: {status}")

DD_API_KEY: d592ba5b751f...
DD_APP_KEY: ddapp_8YZUnW...
DD_SITE: us5.datadogh...
ANTHROPIC_API_KEY: sk-ant-api03...


## 1. Enable LLM Observability

In [13]:
os.environ.setdefault("DD_TRACE_ENABLED", "false")  # no local DD agent

from ddtrace.llmobs import LLMObs

# Replace with your actual project name from Datadog
PROJECT_NAME = "Tax Admin"  # <-- update if different
DATASET_NAME = "Tax Quality Check"

LLMObs.enable(
    ml_app="tax_admin",
    agentless_enabled=True,
    project_name=PROJECT_NAME,
)
print("LLMObs enabled:", LLMObs.enabled)

LLMObs enabled: True


## 2. Pull Dataset from Datadog

The dataset records should have:
- `input_data`: a dict with a `document_text` key (raw text from a tax document)
- `expected_output`: what we expect Claude to classify/extract (e.g. `"W2"`, `"business_deduction"`, a dollar amount)

In [14]:
dataset = LLMObs.pull_dataset(
    dataset_name=DATASET_NAME,
    project_name=PROJECT_NAME,
)

# Limit to 3 records to control cost
dataset = dataset[:3]

print(f'Using {len(dataset)} records')
if len(dataset) > 0:
    print('First record preview:')
    print('  input_data:', str(dataset[0].get('input_data', {}))[:200])
    print('  expected_output:', dataset[0].get('expected_output', '-'))


Dataset: Tax Quality Check
Records: 16

First record preview:
  input_data: [{'content': 'You are an expert tax preparer. Extract financial data from tax documents. Return ONLY raw JSON — no markdown, no code fences, no explanation.', 'role': 'system'}, {'content': 'Partial a
  expected_output: [{'role': 'reasoning'}, {'content': '{\n  "income": [\n    {"description": "W2 wages (Gergely - Datadog & Sequoia One PEO/Paradigm)", "amount": 199108.99, "source": "Datadog, Inc & Sequoia One PEO, LLC", "type": "W2"},\n    {"description": "Self-employment income (Anna - Prada & Dior events)", "amount": 14214.17, "source": "PRADA USA Corp. & Marcus Kan (Dior)", "type": "1099-NEC"},\n    {"description": "Interest income (combined JPMorgan Chase accounts)", "amount": 5271.53, "source": "JPMorgan Chase Bank", "type": "1099-INT"},\n    {"description": "Ordinary dividends (combined)", "amount": 18531.33, "source": "JPMorgan Securities LLC", "type": "1099-DIV"},\n    {"description": "Capital gain dist

In [ ]:
# Inspect actual record structure from Datadog
if len(dataset) > 0:
    rec = dataset[0]
    print('Record type:', type(rec))
    print('Record:', rec)


## 3. (Optional) Add Sample Records to Dataset

Run this cell once to seed the dataset with representative tax document snippets if it's empty.

In [5]:
SAMPLE_RECORDS = [
    {
        "input_data": {
            "document_text": "Form W-2 Wage and Tax Statement 2025. Employer: Datadog Inc. Employee: Gergely Svigruha. Box 1 Wages: $185,000.00. Box 2 Federal income tax withheld: $42,000.00. Box 16 State wages: $185,000.00.",
            "task": "Classify document type and extract total income amount."
        },
        "expected_output": "W2 | $185,000.00",
        "metadata": {"doc_type": "W2", "year": "2025", "difficulty": "easy"}
    },
    {
        "input_data": {
            "document_text": "INVOICE. From: Anna Huang. To: Marcus Kan. Services: Prada Superheroes In-store Event, Saks 5th Ave, New York, March 21-22 2025. 16 hours x $150/hr = $2,400.00. Total due: $2,400.00.",
            "task": "Classify document type and extract total income amount."
        },
        "expected_output": "1099-NEC | $2,400.00",
        "metadata": {"doc_type": "1099-NEC", "year": "2025", "difficulty": "easy"}
    },
    {
        "input_data": {
            "document_text": "The Yard Lincoln Square — Dedicated Desk Membership. Member: Gergely Svigruha. Monthly rate: $950/month. Period: January 2025 - December 2025. Total annual: $11,400. Business use: 100%.",
            "task": "Classify deduction category and extract annual amount."
        },
        "expected_output": "business | $11,400.00",
        "metadata": {"doc_type": "expense", "category": "office", "difficulty": "easy"}
    },
    {
        "input_data": {
            "document_text": "IRS 1040-ES Estimated Tax Payment. Taxpayer: Gergely Svigruha. Tax Period: Q1 2025. Payment amount: $7,500.00. Payment date: April 14, 2025. Jurisdiction: Federal.",
            "task": "Classify as estimated_payment and extract amount and jurisdiction."
        },
        "expected_output": "estimated_payment | federal | $7,500.00",
        "metadata": {"doc_type": "estimated_payment", "jurisdiction": "federal", "difficulty": "easy"}
    },
    {
        "input_data": {
            "document_text": "Delta Air Lines Receipt. Passenger: Anna Huang. Route: JFK-LHR-JFK. Outbound: 19 Feb 2025. Return: 10 Mar 2025. Ticket price: $1,670.51. Purpose: Business travel to London client.",
            "task": "Classify deduction category and extract amount."
        },
        "expected_output": "business | $1,670.51",
        "metadata": {"doc_type": "expense", "category": "travel", "difficulty": "medium"}
    },
    {
        "input_data": {
            "document_text": "State Farm Insurance. Policy: Condominium Unit Owners. Insured: Gergely Svigruha. Annual premium: $1,904.00. Property: 123 Main St, New York NY. Coverage: dwelling, personal property, liability.",
            "task": "Classify deduction category and extract amount. Note: this may or may not be deductible."
        },
        "expected_output": "other | $1,904.00",
        "metadata": {"doc_type": "expense", "category": "insurance", "difficulty": "hard"}
    },
    {
        "input_data": {
            "document_text": "Form 1099-INT Interest Income 2025. Payer: Chase Bank. Recipient: Anna Huang. Box 1 Interest income: $847.32. Account: savings account ending 4521.",
            "task": "Classify document type and extract total income amount."
        },
        "expected_output": "1099-INT | $847.32",
        "metadata": {"doc_type": "1099-INT", "year": "2025", "difficulty": "easy"}
    },
    {
        "input_data": {
            "document_text": "T-Mobile Bill. Account holder: Gergely Svigruha. Billing period: November 2025. Total charges: $166.20. Lines: 2 (personal + business). Business line charges: $83.10.",
            "task": "Classify deduction category and extract the business-use portion amount."
        },
        "expected_output": "business | $83.10",
        "metadata": {"doc_type": "expense", "category": "communications", "difficulty": "hard"}
    },
]

if len(dataset) == 0:
    print("Dataset is empty — adding sample records...")
    dataset.append(SAMPLE_RECORDS)
    dataset.push()
    print(f"Added {len(SAMPLE_RECORDS)} records.")
else:
    print(f"Dataset already has {len(dataset)} records — skipping seed.")

NameError: name 'dataset' is not defined

## 4. Define the Task

The task calls Claude Haiku to classify and extract data from a tax document snippet.

In [15]:
import anthropic
from typing import Any, Optional

anthropic_client = anthropic.Anthropic()

EXTRACTION_SYSTEM = (
    "You are a tax document classifier. Given a tax document excerpt, "
    "extract the key information in exactly the format requested. Be precise with dollar amounts."
)


def _parse_input(input_data: Any) -> tuple[str, str]:
    default_task = "Classify document type and extract total income or deduction amount."
    if isinstance(input_data, dict):
        return input_data.get("document_text", str(input_data)), input_data.get("task", default_task)
    if isinstance(input_data, list):
        return " ".join(str(v) for v in input_data), default_task
    return str(input_data), default_task


def tax_extraction_task(
    input_data: Any,
    config: Optional[dict] = None,
    metadata: Optional[dict] = None,
) -> str:
    config = config or {}
    model = config.get("model", "claude-haiku-4-5-20251001")
    doc_text, task_instruction = _parse_input(input_data)
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=256,
        system=EXTRACTION_SYSTEM,
        messages=[{"role": "user", "content": f"Document:\n{doc_text}\n\nTask: {task_instruction}\n\nFormat: 'type | $amount' or 'type | jurisdiction | $amount'. Nothing else."}],
    )
    return response.content[0].text.strip()


# Smoke test
test = {"document_text": "Form W-2. Employer: Datadog. Wages: $185,000.", "task": "Classify and extract income."}
print("dict input:", tax_extraction_task(test))
print("list input:", tax_extraction_task(["Form W-2. Employer: Datadog. Wages: $185,000."]))


dict input: Wages | $185,000
list input: W-2 | $185,000


## 5. Define Evaluators

In [16]:
import re
from ddtrace.llmobs import LLMJudge, BooleanStructuredOutput, ScoreStructuredOutput


def doc_type_correct(input_data, output_data: str, expected_output: str) -> bool:
    expected_type = expected_output.split('|')[0].strip().lower()
    return expected_type in output_data.lower()


def amount_correct(input_data, output_data: str, expected_output: str) -> bool:
    def extract_amount(s):
        match = re.search(r'\$([\d,]+\.?\d*)', s)
        return float(match.group(1).replace(',', '')) if match else None
    expected_amt = extract_amount(expected_output)
    output_amt = extract_amount(output_data)
    if expected_amt is None or output_amt is None:
        return False
    return abs(expected_amt - output_amt) <= 1.0


def format_valid(input_data, output_data: str, expected_output: str) -> bool:
    return '|' in output_data and '$' in output_data


# LLM-as-a-judge only for overall accuracy — single Haiku call per record
accuracy_judge = LLMJudge(
    provider='anthropic',
    model='claude-haiku-4-5-20251001',
    user_prompt=(
        'You are evaluating a tax document classifier.\n'
        'Model output: {{output_data}}\n'
        'Expected output: {{expected_output}}\n\n'
        'Is the classification and dollar amount both correct?'
    ),
    structured_output=BooleanStructuredOutput(
        description='Classification and amount both correct',
        reasoning=False,
        pass_when=True,
    ),
)

print('Evaluators ready: doc_type_correct, amount_correct, format_valid, accuracy_judge')


Evaluators defined: doc_type_correct, amount_correct, format_valid, accuracy_judge, amount_score_judge


## 6. Run Experiment — Baseline (Haiku)

In [17]:
experiment = LLMObs.experiment(
    name='tax-extraction-haiku',
    task=tax_extraction_task,
    dataset=dataset,
    evaluators=[doc_type_correct, amount_correct, format_valid, accuracy_judge],
    config={'model': 'claude-haiku-4-5-20251001'},
    description='Tax extraction using Claude Haiku — 3 records',
)

result = experiment.run(jobs=1)
print('\nExperiment complete.')
print(result)


failed to send, dropping 2 traces to intake at http://localhost:8126/v0.5/traces: client error (Connect) [6 skipped]


KeyboardInterrupt: 

failed to send, dropping 1 traces to intake at http://localhost:8126/v0.5/traces: client error (Connect) [13 skipped]
failed to send, dropping 1 traces to intake at http://localhost:8126/v0.5/traces: client error (Connect) [7 skipped]


## 7. Run Experiment — Comparison (Sonnet)

In [ ]:
# Sonnet comparison — uncomment when ready to spend more
# experiment_sonnet = LLMObs.experiment(
#     name='tax-extraction-sonnet',
#     task=tax_extraction_task,
#     dataset=dataset,
#     evaluators=[doc_type_correct, amount_correct, format_valid, accuracy_judge],
#     config={'model': 'claude-sonnet-4-6'},
# )
# result_sonnet = experiment_sonnet.run(jobs=1)


## 8. Compare Results

In [ ]:
import json

def summarize(name, result):
    print(f"\n{'='*50}")
    print(f"Experiment: {name}")
    print(f"{'='*50}")
    if hasattr(result, '__dict__'):
        print(json.dumps(result.__dict__, indent=2, default=str))
    else:
        print(result)

summarize("Haiku", result_haiku)
summarize("Sonnet", result_sonnet)

## 9. Add Records to Dataset

Use this cell to expand the dataset with new document samples.

In [ ]:
new_records = [
    {
        "input_data": {
            "document_text": "",  # paste document text here
            "task": "Classify document type and extract total income or deduction amount."
        },
        "expected_output": "",  # e.g. "W2 | $120,000.00"
        "metadata": {"doc_type": "", "difficulty": ""}
    }
]

# Uncomment to add:
# dataset.append(new_records)
# dataset.push()
# print(f"Dataset now has {len(dataset)} records.")